# OpenCLIP embeddings + dimensionality reduction + optional DBSCAN

Pipeline: **CSV → OpenCLIP image embeddings → L2 normalization → PCA → 2D (UMAP / t-SNE / PCA) → plot** colored / shaped by the columns you choose (`sobrenome`, `categoria`, `região`, ...).

## 1. Setup

In [8]:
!pip -q install open_clip_torch umap-learn plotly

In [1]:
!git clone https://github.com/Facct2027Era-so-mais-um-Silva/comparative-catalog.git

Cloning into 'comparative-catalog'...
remote: Enumerating objects: 1459, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 1459 (delta 6), reused 0 (delta 0), pack-reused 1440 (from 1)
Receiving objects: 100% (1459/1459), 709.01 MiB | 17.50 MiB/s, done.
Resolving deltas: 100% (165/165), done.
Updating files: 100% (1220/1220), done.


In [23]:
!wget https://raw.githubusercontent.com/Facct2027Era-so-mais-um-Silva/comparative-catalog/refs/heads/main/codes/samples_en.csv

--2026-10-31 19:29:09--  https://raw.githubusercontent.com/Facct2027Era-so-mais-um-Silva/comparative-catalog/refs/heads/main/codes/samples_en.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23422 (23K) [text/plain]
Saving to: ‘samples_en.csv’

samples_en.csv      100%[===================>]  22.87K  --.-KB/s    in 0s      

2026-10-31 19:29:09 (166 MB/s) - ‘samples_en.csv’ saved [23422/23422]



In [24]:
import os, warnings
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## 2. Configuration

Everything you are likely to change is here.

In [25]:
# ---------------- Data ----------------
# Path to your CSV. If it is not found, you will be asked to upload it.
CSV_PATH = "samples_en.csv"

# Optional row filter, e.g. {"model": ["qwen"]} or {"categoria": ["Elite", "Popular"]}
FILTER = {"model": ['meta']}
SAVE_FILENAME = "clip_meta" # filename assigned to save outputs

# Treat these values as "no value" (the CSV uses "—" for região when it does not apply)
MISSING_LABELS = {"—": "n/a"}

# ---------------- Embedding model ----------------
OPENCLIP_MODEL      = "ViT-L-14"                 # try "ViT-H-14" for a stronger (heavier) model
OPENCLIP_PRETRAINED = "laion2b_s32b_b82k"        # ViT-H-14 -> "laion2b_s32b_b79k"
BATCH_SIZE = 32
CACHE_PATH = "/content/embeddings_cache.npz"     # embeddings are cached here
USE_CACHE  = True                                # set False to force re-embedding

# ---------------- Normalization / reduction ----------------
PCA_DIMS      = 50         # PCA before the 2D reduction (auto-capped by sample size)
REDUCER       = "umap"     # "umap" | "tsne" | "pca"
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST  = 0.1
TSNE_PERPLEXITY = 30
SEED = 42

# ---------------- Visualization ----------------
# Columns available: "sobrenome", "categoria", "região", "genero", "model"
COLOR_BY  = "region"
MARKER_BY = "region"       # second dimension as marker shape; None to disable
POINT_SIZE = 90

## 3. Load the CSV

In [26]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

for col in ["surname", "caregory", "region", "gender", "model"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().replace(MISSING_LABELS)

df['path'] = "comparative-catalog/images/" + df['model'] + '_' + df['id'].astype(str) + '.jpg'
for i, path in enumerate(df['path']):
    if not os.path.isfile(path):
        replaced = False
        for ext in ['.png', '.webp','.jpeg']: #### fix image extension
            new_file = path[:path.rfind('.')] + ext
            if os.path.isfile(new_file):
                replaced = True
                df['path'].iloc[i] = new_file
        if not replaced: #### qwen sometimes generates two images for a single prompt
            df['path'].iloc[i] = df['path'].iloc[i][:-4] + '_1.png'

        if not os.path.isfile(df['path'].iloc[i]): print(df['path'].iloc[i])

### Filters
for col, keep in FILTER.items():
    df = df[df[col].isin(keep)]
df = df.reset_index(drop=True)
df

,model,id,surname,category,gender,region,path
0,meta,1,Silva,Popular,man,n/a,comparative-catalog/images/meta_1.webp
1,meta,2,Silva,Popular,man,North,comparative-catalog/images/meta_2.webp
2,meta,3,Silva,Popular,man,Northeast,comparative-catalog/images/meta_3.webp
3,meta,4,Silva,Popular,man,Central-West,comparative-catalog/images/meta_4.webp
4,meta,5,Silva,Popular,man,Southeast,comparative-catalog/images/meta_5.webp
...,...,...,...,...,...,...,...
193,meta,194,(no surname),Regional control,person,North,comparative-catalog/images/meta_194.jpg
194,meta,195,(no surname),Regional control,person,Northeast,comparative-catalog/images/meta_195.webp
195,meta,196,(no surname),Regional control,person,Central-West,comparative-catalog/images/meta_196.webp
196,meta,197,(no surname),Regional control,person,Southeast,comparative-catalog/images/meta_197.webp


## 4. OpenCLIP embeddings

In [9]:
import open_clip

def embed_images(paths):
    model, _, preprocess = open_clip.create_model_and_transforms(
        OPENCLIP_MODEL, pretrained=OPENCLIP_PRETRAINED, device=device)
    model.eval()
    feats = []
    with torch.no_grad(), torch.autocast(device_type=device, enabled=(device == "cuda")):
        for i in tqdm(range(0, len(paths), BATCH_SIZE), desc="Embedding"):
            batch = torch.stack([preprocess(Image.open(p).convert("RGB"))
                                 for p in paths[i:i + BATCH_SIZE]]).to(device)
            feats.append(model.encode_image(batch).float().cpu().numpy())
    return np.concatenate(feats)

cache_key = np.array([OPENCLIP_MODEL, OPENCLIP_PRETRAINED] + list(df["path"]))
E = None
if USE_CACHE and os.path.exists(CACHE_PATH):
    z = np.load(CACHE_PATH, allow_pickle=True)
    if np.array_equal(z["key"], cache_key):
        E = z["emb"]
        print("Loaded embeddings from cache")
if E is None:
    E = embed_images(list(df["path"]))
    np.savez(CACHE_PATH, emb=E, key=cache_key)

print("raw embeddings:", E.shape)

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

Embedding:   0%|          | 0/7 [00:00<?, ?it/s]

raw embeddings: (198, 768)


## 5. Normalization and dimensionality reduction

1. **L2 normalization** (unit vectors, so Euclidean distance ≈ cosine distance)
2. **PCA** to `PCA_DIMS` dimensions
3. **2D reduction** with UMAP / t-SNE / PCA

In [10]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize

n = len(df)
X_proc = normalize(E, norm="l2")                      # L2 normalization

n_pca = max(2, min(PCA_DIMS, n - 1, X_proc.shape[1]))
X_pca = PCA(n_components=n_pca, random_state=SEED).fit_transform(X_proc)

def reduce_2d(X, method):
    if method == "pca":
        return PCA(n_components=2, random_state=SEED).fit_transform(X)
    if method == "tsne":
        from sklearn.manifold import TSNE
        perp = max(2, min(TSNE_PERPLEXITY, (n - 1) / 3))
        return TSNE(n_components=2, perplexity=perp, metric="cosine", init="pca",
                    random_state=SEED).fit_transform(X)
    if method == "umap":
        import umap
        return umap.UMAP(n_components=2, n_neighbors=max(2, min(UMAP_NEIGHBORS, n - 1)),
                         min_dist=UMAP_MIN_DIST, metric="cosine", random_state=SEED).fit_transform(X)
    raise ValueError(method)

X2 = reduce_2d(X_pca, REDUCER)
df["x"], df["y"] = X2[:, 0], X2[:, 1]
print(f"PCA dims: {n_pca} | explained variance: {PCA(n_components=n_pca, random_state=SEED).fit(X_proc).explained_variance_ratio_.sum():.1%} | 2D method: {REDUCER}")

PCA dims: 50 | explained variance: 88.4% | 2D method: umap


## 6. Plot

Color = `COLOR_BY`, marker shape = `MARKER_BY`.

### Interactive (with HTML export)

In [20]:
import io, json, base64
import plotly.express as px
from PIL import Image
from IPython.display import HTML, display

# Two similar pairs (blue, warm) + two distinct colors, from Okabe-Ito
COLORS = ["#0072B2", "#56B4E9", "#D55E00", "#E69F00", "#009E73", "#CC79A7"]
SYMBOLS = ["circle", "square", "diamond", "triangle-up", "cross", "x",
           "star", "hexagon", "triangle-down"]
REGION_ORDER = ["North", "Northeast", "Central-West", "Southeast", "South", "n/a"]


def make_thumbnail(path, size=160, quality=80):
    """Return a small JPEG as a base64 data URI."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def build_maps(d, color_by):
    """Category order + explicit color/symbol maps ("n/a" -> neutral gray)."""
    if color_by == "region":
        cats = [c for c in REGION_ORDER if c in set(d[color_by].astype(str))]
    else:
        cats = sorted(d[color_by].dropna().astype(str).unique())
    color_map, symbol_map, i = {}, {}, 0
    for cat in cats:
        color_map[cat] = COLORS[i % len(COLORS)]
        symbol_map[cat] = SYMBOLS[i % len(SYMBOLS)]
        i += 1
    return cats, color_map, symbol_map


def plot_with_thumbnails(df, color_by,
                         info_cols=("surname", "category", "region", "gender"),
                         thumb_size=160, width=950, height=650, title=None):
    d = df.copy()
    d[color_by] = d[color_by].astype(str)          # treat as discrete categories
    info_cols = [c for c in info_cols if c in d.columns]
    if color_by not in info_cols:                  # tooltip always names the group in text
        info_cols.insert(0, color_by)
    d["thumb"] = [make_thumbnail(p, thumb_size) for p in d["path"]]

    cats, color_map, symbol_map = build_maps(d, color_by)

    # customdata[0] = thumbnail, customdata[1:] = the info columns
    fig = px.scatter(
        d, x="x", y="y",
        color=color_by, symbol=color_by,           # same column -> color AND shape
        color_discrete_map=color_map, symbol_map=symbol_map,
        category_orders={color_by: cats},
        custom_data=["thumb"] + info_cols,
        width=width, height=height,
    )
    fig.update_traces(marker=dict(size=14, line=dict(width=1.2, color="black")),
                      hoverinfo="none")            # custom tooltip below; events still fire
    fig.update_layout(
        title=title,
        font=dict(size=15),
        legend=dict(font=dict(size=15), title_font=dict(size=15), itemsizing="constant"),
        plot_bgcolor="white",
    )
    fig.update_xaxes(showgrid=True, gridcolor="#E5E5E5", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#E5E5E5", zeroline=False)

    post_script = """
    var gd  = document.getElementById('{plot_id}');
    var cols = __COLS__;
    var tip = document.createElement('div');
    tip.style.cssText = 'position:fixed;display:none;pointer-events:none;z-index:9999;' +
        'background:#fff;border:1px solid #888;border-radius:6px;padding:6px;' +
        'box-shadow:0 2px 8px rgba(0,0,0,.3);font:14px sans-serif;color:#222;';
    document.body.appendChild(tip);

    gd.on('plotly_hover', function(ev) {
        var cd = ev.points[0].customdata;
        var html = '<img src="' + cd[0] + '" style="display:block;max-width:__SIZE__px;margin-bottom:4px">';
        for (var i = 0; i < cols.length; i++) {
            html += '<div><b>' + cols[i] + ':</b> ' + cd[i + 1] + '</div>';
        }
        tip.innerHTML = html;
        tip.style.display = 'block';
        var e = ev.event, x = e.clientX + 15, y = e.clientY + 15;
        // keep the tooltip inside the window
        if (x + tip.offsetWidth  > window.innerWidth)  x = e.clientX - tip.offsetWidth  - 15;
        if (y + tip.offsetHeight > window.innerHeight) y = e.clientY - tip.offsetHeight - 15;
        tip.style.left = x + 'px';
        tip.style.top  = y + 'px';
    });
    gd.on('plotly_unhover', function() { tip.style.display = 'none'; });
    """.replace("__COLS__", json.dumps(info_cols)).replace("__SIZE__", str(thumb_size))

    html = fig.to_html(include_plotlyjs="cdn", full_html=True, post_script=post_script)
    display(HTML(html))

    with open(f"{SAVE_FILENAME}.html", "w", encoding="utf-8") as f:
        f.write(html)
    return fig

filter_title = ",".join([f"{key.capitalize()}: {",".join([v.capitalize() for v in value])}" for key, value in FILTER.items()])
fig = plot_with_thumbnails(
    df, COLOR_BY,
    title=f"{REDUCER.upper()} of OpenCLIP embeddings | {filter_title}",
)

### PDF Export

In [21]:
import matplotlib.pyplot as plt

MPL_MARKERS = {"circle": "o", "square": "s", "diamond": "D", "triangle-up": "^",
               "cross": "P", "x": "X", "star": "*", "hexagon": "h", "triangle-down": "v"}

d = df.copy()
d[COLOR_BY] = d[COLOR_BY].astype(str)
cats, color_map, symbol_map = build_maps(d, COLOR_BY)

plt.rcParams["pdf.fonttype"] = 42          # embed fonts as real text
fig_pdf, ax = plt.subplots(figsize=(9.5, 6.5))

for cat in cats:
    sub = d[d[COLOR_BY] == cat]
    ax.scatter(sub["x"], sub["y"],
               c=color_map[cat], marker=MPL_MARKERS[symbol_map[cat]],
               s=110, edgecolors="black", linewidths=1.0, label=cat)

ax.grid(True, color="#E5E5E5")
ax.set_axisbelow(True)

filter_title = ",".join([f"{key.capitalize()}: {",".join([v.capitalize() for v in value])}" for key, value in FILTER.items()])
ax.set_title(f"{REDUCER.upper()} of OpenCLIP embeddings | {filter_title}", fontsize=14)

ax.legend(title=COLOR_BY, fontsize=12, title_fontsize=12, frameon=True)
ax.legend(title=COLOR_BY, fontsize=12, title_fontsize=12, frameon=True,
          loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)
fig_pdf.tight_layout()
fig_pdf.savefig(f"{SAVE_FILENAME}.pdf", bbox_inches="tight")   # keeps the outside legend from being cropped
plt.close(fig_pdf)